In [20]:
import pandas as pd
import numpy as np
import chromadb
from groq import Groq
import os
from dotenv import load_dotenv
from scipy.special import softmax
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
df = pd.read_csv("../data/cleaned/cleaned_reviews.csv")
embeddings = np.load("../data/cleaned/sentence_embeddings.npy")

In [8]:
chroma_client = chromadb.PersistentClient(path="../data/chromadb")
collection = chroma_client.get_or_create_collection(
    name="complaint_embeddings",
    metadata={"hnsw:space":"cosine"} 
    ) 

In [11]:
import re
import emoji
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
def text_preprocessing(text,remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = re.sub(r'&[a-z]+;|&#\d+;', '', text) 
    text = re.sub(r'<[^>]+>', '', text)
    text = emoji.replace_emoji(text,replace="")
    special_character_pattern = r'[^a-zA-Z0-9\s]+'
    text = re.sub(special_character_pattern,"",text)
    text = re.sub(r"\s+"," ",text).strip()
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        words = text.split(" ")
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
    
    return text

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def retrieve_filtered(query,collection,model,topic_label,top_k=3):
    query = text_preprocessing(query,remove_stopwords=False)
    query_embedding = model.encode(query).tolist()
    result = collection.query(
        query_embeddings = [query_embedding],
        n_results=top_k,
        include= ['documents','metadatas','distances'],
        where= {"topic_label":{"$eq": topic_label}}
    )
    return result['documents'][0]

In [22]:
topic_centroids = {}

for topic_id in df['topic_id'].unique():
    if topic_id == -1:
        continue
    
    mask = df['topic_id'] == topic_id
    topic_centroids[topic_id] = embeddings[mask].mean(axis=0)

In [23]:
topic_ids_sorted = sorted(topic_centroids.keys())
centroid_matrix = np.array([topic_centroids[t_id] for t_id in topic_ids_sorted])

In [18]:
topic_labels_sorted = [
    "Charging issues",
    "Ring/Card Slot Cases",
    "Wearable bands",
    "Phone cases",
    "Phone issue",
    "Case Fitting issues", 
    "Mounts,Holder issue",
    "Fingerprints/screen protector",
    "Cracked screen/Display",
    "Broken item", 
    "Item stopped working",
    "Case Color related",
    "Sound/Hearing issues",
    "Issue with buttons/Buttons not working",
    "Camera lens/protector issues",
    "Screen protector issue (bubble formation)",
    "Fitting issues", 
    "Generic wear/durability complaints",
    "Vent mounts/holder issues",
    "Item does not work", 
    "Return related issue",
    "Scratch issues",
    "Popsockets",
    "Build Quality/Form Factor complaints",
    "Product Quality issues",
    "Glass Screen related issues",
    "Irrelevant",
    "Phone Covers/Protection",
    "Irrelevant",
    "Irrelevant",
    "Irrelevant",
    "Tripod issues",
    "Plastic product",
    "Not a genuine product",
    "Crystals lost",
    "Not reliable",
    "Irrelevant",
    "Installation issues",
    "Device Fitting issues" 
]

In [21]:
def predict_proba(texts):
    embeddings_batch = model.encode(texts) # shape (n,384)
    similarities = cosine_similarity(embeddings_batch,centroid_matrix) # shape (n,39)
    probabilities = softmax(similarities,axis=1) # shape (n,39)
    return probabilities

In [19]:
load_dotenv("../apikey.env")
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
def generate_response(complaint,topic_label,context=[]):
    context_text = "\n".join([f"{i+1}.{c}" for i,c in enumerate(context)])
    if not context:
        prompt = f"Topic: {topic_label}\nComplaint: {complaint}"
    else:
        prompt = f"Topic:{topic_label}\nComplaint: {complaint}\nSimilar past complaints: {context_text}"

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role":"system",
                "content": "You are a helpful customer support agent for an electronics accessories company.Your job is to write a professional, empathetic reply to a customer complaint.Keep your response concise and actionable."
            },
            {
                "role":"user",
                "content": prompt
            }
        ],
        model= "llama-3.1-8b-instant"
    )
    return chat_completion.choices[0].message.content